# Data Preprocessing for Task 3: Sentiment Classification

## Importing libraries

In [15]:
# ! pip install pandas openpyxl emoji nltk datasets textblob

In [16]:
import re
import unicodedata
import numpy as np
import pandas as pd
import emoji
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import word_tokenize, pos_tag
from datasets import load_dataset
from textblob import TextBlob

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

True

## Dataset ingestion

In [17]:
path = "../data/raw-data/corpus_full.xlsx"
df = pd.read_excel(path, engine="openpyxl")
print(f"Loaded {path} | rows: {len(df)}")

Loaded ../data/raw-data/corpus_full.xlsx | rows: 10527


In [ ]:
texts = df["text"].astype(str)

print(f"Rows: {len(df)}")
print(f"Non-empty text: {texts.str.strip().str.len().gt(0).sum()}")
print(f"Avg chars: {texts.str.len().mean():.0f}")
print(f"Avg words: {texts.str.split().str.len().mean():.0f}")


Rows: 10527
Non-empty text: 10527
Avg chars: 442
Avg words: 79


In [ ]:
emoji_counter = {}
for s in texts:
    for item in emoji.emoji_list(s):
        ch = item["emoji"]
        emoji_counter[ch] = emoji_counter.get(ch, 0) + 1
        
if not emoji_counter:
    print("No unicode emojis found in dataset.")
else:
    emoji_df = pd.DataFrame(
        sorted(emoji_counter.items(), key=lambda kv: kv[1], reverse=True),
        columns=["emoji", "count"],
    )
    print(f"Unique emojis: {len(emoji_df)}")
    emoji_df["demojize"] = emoji_df["emoji"].map(lambda e: emoji.demojize(e))
    display(emoji_df.head(20))

Unique emojis: 4


,emoji,count,demojize
0,🆘,1,:SOS_button:
1,®,1,:registered:
2,1⃣,1,:keycap_1:
3,2⃣,1,:keycap_2:


## Preprocessing functions
- For mircotext normalization, we use light slang expansion only by setting strict ALLOWLIST in _load_abbrev_patterns.
- Without creating an ALLOWLIST, Hugging Face maps texts like "so" --> "significant other".

In [ ]:
# Remove HN quote markers ('>') while retaining usage in expressions e.g., A>B or A>=B
def remove_hn_blockquotes(text):
    lines = [line.lstrip(">").strip() for line in text.splitlines() if line.strip()]
    t = " ".join(lines)
    t = re.sub(r"\s>\s", " ", t)
    return re.sub(r"\s+", " ", t).strip()

# Remove markdown bold (**), italic (*), inline code (`), links [text](url)
def remove_markdown_formatting(text):
    t = text
    t = re.sub(r"\*\*(.+?)\*\*", r"\1", t)
    t = re.sub(r"\*(.+?)\*", r"\1", t)
    t = re.sub(r"`(.+?)`", r"\1", t)
    t = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", t)
    return t

# Expand contractions
CONTRACTIONS = {
    "don't": "do not", "doesn't": "does not", "didn't": "did not", "won't": "will not",
    "can't": "cannot", "couldn't": "could not", "wouldn't": "would not", "shouldn't": "should not",
    "isn't": "is not", "aren't": "are not", "wasn't": "was not", "weren't": "were not",
    "haven't": "have not", "hasn't": "has not", "hadn't": "had not", "mustn't": "must not",
    "i'm": "i am", "you're": "you are", "he's": "he is", "she's": "she is", "it's": "it is",
    "we're": "we are", "they're": "they are", "i've": "i have", "you've": "you have",
    "we've": "we have", "they've": "they have", "i'd": "i would", "you'd": "you would",
    "he'd": "he would", "she'd": "she would", "we'd": "we would", "they'd": "they would",
    "i'll": "i will", "you'll": "you will", "he'll": "he will", "she'll": "she will",
    "we'll": "we will", "they'll": "they will", "that's": "that is", "there's": "there is",
    "here's": "here is", "what's": "what is", "who's": "who is", "let's": "let us",
    "that'll": "that will", "there'll": "there will", "this'll": "this will",
}

def expand_contractions(text):
    t = text
    for cont, exp in CONTRACTIONS.items():
        t = re.sub(re.escape(cont), exp, t, flags=re.IGNORECASE)
    return t

# Load abbreviations except common/device tokens
def _load_abbrev_patterns():
    BLOCKED_KEYS = frozenset({
        # common words
        "so", "as", "or", "if", "is", "it", "in", "on", "at", "to", "of", "and",
        "the", "a", "an", "i", "you", "we", "they", "he", "she", "me", "my",
        "be", "do", "go", "no", "up", "by", "us",
        # device/tech
        "ios", "android", "iphone", "ipad", "macos", "usb", "usb-c", "lte", "nfc",
        "cpu", "gpu", "ram", "rom", "oled", "hdr", "heic", "jpeg", "raw", "av1",
        "api", "ui", "ux", "aosp", "oem", "arm", "tsmc",
    })

    ds = load_dataset("willwade/txt-sms-abbreviations", split="train")

    # Normalize key for abbreviation lookups by lowercasing and removing underscores
    def _norm_key(a):
        return a.lower().replace("_", "")

    # Avoids expanding blocked keys + device/acronym/model tokens
    def _is_protected_token(a, low_a):
        core = a.replace("_", "")
        if low_a in BLOCKED_KEYS:
            return True
        # Protect model tokens e.g., S23, 128GB
        if any(ch.isalpha() for ch in core) and any(ch.isdigit() for ch in core):
            return True
        # Protect common uppercase acronyms
        if 2 <= len(core) <= 6 and core.isupper():
            return True
        return False
    by_key = {}

    # Iterate through abbreviation dataset and build mapping for expansion
    for row in ds:
        a = str(row.get("Abbreviation", "") or "").strip()
        e = str(row.get("Expansion", "") or "").strip()
        if not a or not e:
            continue
        if len(a) == 1 and a.isdigit():
            continue
        low_a = _norm_key(a)
        if _is_protected_token(a, low_a):
            continue

        rep = f" {e} "
        # Take first occurrence of abbreviation for mapping
        if low_a not in by_key:
            by_key[low_a] = (a, rep)

    mapping = list(by_key.values())

    # Sort so that longer abbreviations match first to prevent substring issues
    mapping.sort(key=lambda x: -len(x[0]))

    # Construct regex patterns for each abbreviation
    patterns = []
    for a, rep in mapping:
        esc = re.escape(a)
        # Use word boundaries for alphanumeric abbreviations, otherwise, use custom boundaries
        pat = (r"\b" + esc + r"\b") if a.replace("_", "").isalnum() else (r"(?<!\w)" + esc + r"(?!\w)")
        patterns.append((pat, rep))
    return patterns

ABBREV_PATTERNS = _load_abbrev_patterns()

# Function to normalize microtext
def normalize_microtext(text):
    t = str(text or "")

    # Expand forms e.g., "2yrs" -> "2 years"
    def _years_repl(m):
        n = m.group(1)
        return f"{n} year" if n == "1" else f"{n} years"

    # Expand forms e.g., "3mos" -> "3 months"
    def _months_repl(m):
        n = m.group(1)
        return f"{n} month" if n == "1" else f"{n} months"

    t = re.sub(r"\b(\d+)\s*yr(s)?\b", _years_repl, t, flags=re.IGNORECASE)
    t = re.sub(r"\b(\d+)\s*mo(s)?\b", _months_repl, t, flags=re.IGNORECASE)

    for pat, rep in ABBREV_PATTERNS:
        t = re.sub(pat, rep, t, flags=re.IGNORECASE)

    # Reduce character elongation e.g., "sooo coool" -> "soo cool"
    t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    return t

# Replace emoticons with their corresponding words
EMOTICON_MAP = [
    (r":\)", " happy "), (r":\(", " sad "), (r":D", " happy "),
    (r">_<", " frustrated "), (r"<3", " love "),
]

def replace_emoticons(text):
    t = text
    for pat, rep in EMOTICON_MAP:
        t = re.sub(pat, rep, t)
    return t

# Replace emojis with their corresponding words
EMOJI_MAP = {
    "🆘": " distress ",
    "®": " registered ",
    "1⃣": " one ",
    "2⃣": " two "
}

def replace_emojis(text):
    t = text
    for em, rep in EMOJI_MAP.items():
        t = t.replace(em, rep)
    return t

def remove_urls(text):
    return re.sub(r"https?://\S+|www\.\S+", " ", text)

def remove_hashtags(text):
    return re.sub(r"#\w+", " ", text)

def remove_handles(text):
    return re.sub(r"@\w+", " ", text)

# Normalize percent expressions e.g., "10%" -> "10 percent"
_percent_re = re.compile(r"(?<!\\w)(\\d+(?:\\.\\d+)?)\\s*%(?!\\w)")

def normalize_percent(text):
    return _percent_re.sub(r"\\1 percent", text)

def correct_spelling(text):
  return str(TextBlob(text).correct())

# Remove special characters e.g., accents, punctuation
def remove_special_chars(text):
    t = unicodedata.normalize("NFKD", text)
    t = "".join(c for c in t if not unicodedata.combining(c))
    t = re.sub(r"[^\w\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

# Lemmatize text
_lemmatizer = WordNetLemmatizer()

def _get_wordnet_pos(tag):
    if tag.startswith("J"): return wordnet.ADJ
    if tag.startswith("V"): return wordnet.VERB
    if tag.startswith("N"): return wordnet.NOUN
    if tag.startswith("R"): return wordnet.ADV
    return wordnet.NOUN

def lemmatize_text(text):
    tokens = word_tokenize(text)     # Tokenizes input text, tags each word with its POS
    tags = pos_tag(tokens)      # then lemmatizes each token using its POS tag
    return " ".join(_lemmatizer.lemmatize(w, _get_wordnet_pos(t)) for w, t in tags)

def normalize_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()

def preprocess_for_traditional_ml(text, use_spell=True, use_lemma=True):
    t = str(text or "").strip()
    t = remove_hn_blockquotes(t)
    t = remove_markdown_formatting(t)
    t = remove_urls(t)
    t = remove_hashtags(t)
    t = remove_handles(t)
    t = replace_emoticons(t)
    t = replace_emojis(t)
    t = expand_contractions(t)
    t = normalize_microtext(t)
    t = normalize_percent(t)
    t = t.lower()
    t = remove_special_chars(t)
    if use_spell:
      t = correct_spelling(t)
    if use_lemma:
        t = lemmatize_text(t)
    return normalize_whitespace(t)

def preprocess_for_transformers(text):
    t = str(text or "").strip()
    t = remove_hn_blockquotes(t)
    t = remove_markdown_formatting(t)
    t = remove_urls(t)
    t = remove_hashtags(t)
    t = remove_handles(t)
    t = replace_emoticons(t)
    t = replace_emojis(t)
    t = expand_contractions(t)
    t = normalize_microtext(t)
    return normalize_whitespace(t)

## Apply preprocessing

In [ ]:
USE_LEMMATIZATION = True
USE_SPELL_CORRECTION = False

df_clean = df.copy()
df_clean["text_ml"] = df_clean["text"].apply(
    lambda x: preprocess_for_traditional_ml(
        x, use_spell=USE_SPELL_CORRECTION, use_lemma=USE_LEMMATIZATION
    )
)
df_clean["text_ml_no_lemma"] = df_clean["text"].apply(
    lambda x: preprocess_for_traditional_ml(
        x, use_spell=USE_SPELL_CORRECTION, use_lemma=False
    )
)
df_clean["text_transformer"] = df_clean["text"].apply(preprocess_for_transformers)
print(f"Rows: {len(df_clean)}")

Rows: 10527


In [39]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10527 entries, 0 to 10526
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   doc_id            10527 non-null  object        
 1   object_id         10527 non-null  int64         
 2   device            10527 non-null  object        
 3   title             10527 non-null  object        
 4   query             10527 non-null  object        
 5   created_at        10527 non-null  datetime64[ns]
 6   author            10527 non-null  object        
 7   tags              10527 non-null  object        
 8   source_url        9955 non-null   object        
 9   text              10527 non-null  object        
 10  deleted           10527 non-null  bool          
 11  descendants       10527 non-null  int64         
 12  score             10527 non-null  int64         
 13  item_type         10527 non-null  object        
 14  text_ml           1052

In [44]:
col = "text_transformer"

emoji_counter = {}
for s in df_clean[col].astype(str):
    for item in emoji.emoji_list(s):
        ch = item["emoji"]
        emoji_counter[ch] = emoji_counter.get(ch, 0) + 1

if not emoji_counter:
    print(f"No unicode emojis found in df_clean[{col!r}].")
else:
    emoji_df = pd.DataFrame(
        sorted(emoji_counter.items(), key=lambda kv: kv[1], reverse=True),
        columns=["emoji", "count"],
    )
    print(
        f"df_clean[{col!r}] | Unique emojis: {len(emoji_df)}"
    )
    emoji_df["demojize"] = emoji_df["emoji"].map(emoji.demojize)
    display(emoji_df.head(20))

No unicode emojis found in df_clean['text_transformer'].


In [25]:
text_col = "text"  
ml_col = "text_ml"
ml_no_lemma_col = "text_ml_no_lemma"
tr_col = "text_transformer"

cols = [c for c in [text_col, ml_col, ml_no_lemma_col, tr_col] if c in df_clean.columns]
df_clean[cols].sample(8, random_state=42)

,text,text_ml,text_ml_no_lemma,text_transformer
9289,Apple supports right-to-repair bill My (OLED) ...,apple support right to repair bill my oled pix...,apple supports right to repair bill my oled pi...,Apple supports right-to-repair bill My (OLED) ...
7470,Seems like Win8 was them trying to push the mo...,seem like win8 be them try to push the mobile ...,seems like win8 was them trying to push the mo...,Seems like Win8 was them trying to push the mo...
5551,"Using the Galaxy S24 Ultra every day, I keep f...",use the galaxy s24 ultra every day i keep feel...,using the galaxy s24 ultra every day i keep fe...,"Using the Galaxy S24 Ultra every day, I keep f..."
5130,The iPhone 15 Pro is a weird one for me becaus...,the iphone 15 pro be a weird one for me becaus...,the iphone 15 pro is a weird one for me becaus...,The iPhone 15 Pro is a weird one for me becaus...
5841,Been on the iPhone 14 Pro Max for a few weeks ...,be on the iphone 14 pro max for a few week and...,been on the iphone 14 pro max for a few weeks ...,Been on the iPhone 14 Pro Max for a few weeks ...
1688,"Emergency SOS via satellite ""Band n53"" has bee...",emergency so via satellite band n53 have be wi...,emergency sos via satellite band n53 has been ...,"Emergency SOS via satellite ""Band n53"" has bee..."
2271,Why the 2% inflation target? (2023) Well if we...,why the 2 inflation target 2023 well if we be ...,why the 2 inflation target 2023 well if we are...,Why the 2% inflation target? (2023) Well if we...
4477,"Honestly the iPhone 15 feels great at times, b...",honestly the iphone 15 feel great at time but ...,honestly the iphone 15 feels great at times bu...,"Honestly the iPhone 15 feels great at times, b..."


In [31]:
for _, row in df_clean.sample(3, random_state=1).iterrows():
    print()
    print("Original:", row["text"])
    print("ML (lemma):", row["text_ml"])
    print("ML (no lemma):", row["text_ml_no_lemma"])
    print("Transformer:", row["text_transformer"])


Original: I want an iPhone Mini-sized Android phone If you can make a phone I love as much as my Pebbles, I'll buy nothing else, forever. I guess confirming there's a market for it is the first step. My phone requirements haven't changed much in the last ten years. I bought one of the first "phablet" phones with a comically oversized 5" screen that got me ribbed by friends ("compensating for something eh mate?") Now 5" is at the bottom of the available size range. I'm a smallish person/manlet and don't need a phone I'm going to drop, but I do need something big enough that I can reliably type on it. I'll gladly support your endeavor. Thanks for taking the initiative. edit: fine with me to make the phone thicc so it has a day+ battery life. A little thicker is far easier to hold on to, anyway! edit2: I did own a Pixel 6 non-XL for about a day. It was large, but the bigger problem was that I found it incredibly topheavy, which made it difficult to hold on to. I swapped for a used 4a 5G,

In [ ]:
# Save full dataset after preprocessing
out_parquet = "data/preprocessed-data/corpus_preprocessed.parquet"
df_clean.to_parquet(out_parquet, index=False)
print(f"Saved {out_parquet} | rows: {len(df_clean)} | cols: {len(df_clean.columns)}")

# Save to CSV
out_csv = "data/preprocessed-data/corpus_preprocessed.csv"
df_clean.to_csv(out_csv, index=False, encoding="utf-8-sig")
print(f"Saved {out_csv}")

Saved corpus_preprocessed.parquet | rows: 10527 | cols: 17
Saved corpus_preprocessed.csv
